# Ranking Stage: XGBoost

This notebook:
1. Loads ratings and feature tables
2. Generates ranking training set (positive + negative samples)
3. Joins with feature store
4. Trains XGBoost ranking model
5. Saves preprocessing pipeline + model


In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.metrics import ndcg_score
from surprise import SVD, Dataset, Reader
import xgboost as xgb

# Set up paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

CONFIG = {
    "mf": {
        "n_factors": 64,
        "n_epochs": 30,
        "lr_all": 0.005,
        "reg_all": 0.02,
        "random_state": 42,
    },
    "retrieval": {
        "topN": 50,
        "n_random_neg": 15,  # reduced for faster candidate building
    },
    "ranker": {
        "objective": "rank:pairwise",
        "eval_metric": "ndcg@10",
        "learning_rate": 0.05,
        "n_estimators": 2000,
        "early_stopping_rounds": 100,
        "max_depth": 6,
        "min_child_weight": 20,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_lambda": 1.0,
        "random_state": 42,
    },
}

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed directory: {PROCESSED_DIR}")
print(f"Models directory: {MODELS_DIR}")
print(f"MF config: {CONFIG['mf']}")
print(f"Ranker config: {CONFIG['ranker']}")


Project root: /Users/charukagunawardhane/Documents/Development/ML_DL/LocalEnv-recommendation-system
Processed directory: /Users/charukagunawardhane/Documents/Development/ML_DL/LocalEnv-recommendation-system/data/processed
Models directory: /Users/charukagunawardhane/Documents/Development/ML_DL/LocalEnv-recommendation-system/models
MF config: {'n_factors': 64, 'n_epochs': 30, 'lr_all': 0.005, 'reg_all': 0.02, 'random_state': 42}
Ranker config: {'objective': 'rank:pairwise', 'eval_metric': 'ndcg@10', 'learning_rate': 0.05, 'n_estimators': 2000, 'early_stopping_rounds': 100, 'max_depth': 6, 'min_child_weight': 20, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 1.0, 'random_state': 42}


In [2]:
# Load data
ratings_path = PROCESSED_DIR / "ratings_clean.parquet"
print("Loading data...")
df_ratings = pd.read_parquet(ratings_path)

# Ensure timestamp exists
assert 'timestamp' in df_ratings.columns, "ratings_clean must include 'timestamp'"

# Time-based split
df_ratings = df_ratings.sort_values('timestamp').reset_index(drop=True)
split_idx = int(len(df_ratings) * 0.8)
train_df = df_ratings.iloc[:split_idx].copy()
holdout_df = df_ratings.iloc[split_idx:].copy()

# Split holdout into ranker train/val (by time)
holdout_mid = len(holdout_df) // 2
rank_train_df = holdout_df.iloc[:holdout_mid].copy()
rank_val_df = holdout_df.iloc[holdout_mid:].copy()

print(f"Train shape: {train_df.shape}")
print(f"Rank-train shape: {rank_train_df.shape}, Rank-val shape: {rank_val_df.shape}")
print(f"Users in train: {train_df['user_id'].nunique()}, items in train: {train_df['parent_asin'].nunique()}")


Loading data...
Train shape: (561222, 4)
Rank-train shape: (70153, 4), Rank-val shape: (70153, 4)
Users in train: 510379, items in train: 92369


In [3]:
# Compute user/item aggregates on train split
print("Computing user/item aggregates on train split...")
user_stats = train_df.groupby('user_id')['rating'].agg(['count', 'mean']).reset_index()
user_stats.columns = ['user_id', 'user_num_ratings', 'user_mean_rating']

item_stats = train_df.groupby('parent_asin')['rating'].agg(['count', 'mean']).reset_index()
item_stats.columns = ['parent_asin', 'item_num_ratings', 'item_mean_rating']

print(f"User stats shape: {user_stats.shape}, Item stats shape: {item_stats.shape}")


Computing user/item aggregates on train split...
User stats shape: (510379, 3), Item stats shape: (92369, 3)


In [4]:
# Load pre-trained MF model (from retrieval notebook)
import pickle

mf_artifact_path = MODELS_DIR / "svd_surprise.pkl"
with open(mf_artifact_path, "rb") as f:
    mf_artifact = pickle.load(f)

mf_model = mf_artifact['model']
mf_trainset = mf_artifact['trainset']
items_pool_all = mf_artifact.get('items_pool', mf_artifact.get('all_items', []))
user_seen_train = mf_artifact.get('user_seen_items', {})

# Limit items_pool to top 5000 popular items from train_df
popular_items = (
    train_df['parent_asin']
    .value_counts()
    .head(5000)
    .index
    .tolist()
)
items_pool = [asin for asin in popular_items if asin in set(items_pool_all)]

print(f"Loaded MF artifact from {mf_artifact_path}")
print(f"Trainset users: {mf_trainset.n_users}, items: {mf_trainset.n_items}, ratings: {mf_trainset.n_ratings}")
print(f"Items pool size (top5000 popular): {len(items_pool)}; Users with interactions: {len(user_seen_train)}")


def predict_mf(model, trainset, user_id: str, item_id: str):
    """Predict rating with MF; fallback to global mean for unseen ids."""
    try:
        uid = trainset.to_inner_uid(user_id)
        iid = trainset.to_inner_iid(item_id)
        return model.predict(uid, iid).est
    except ValueError:
        return trainset.global_mean


Loaded MF artifact from /Users/charukagunawardhane/Documents/Development/ML_DL/LocalEnv-recommendation-system/models/svd_surprise.pkl
Trainset users: 631986, items: 115709, ratings: 701528
Items pool size (top5000 popular): 5000; Users with interactions: 510379


In [5]:
# Label lookup for holdout slices
def build_label_lookup(split_df: pd.DataFrame):
    return {(row.user_id, row.parent_asin): row.rating for _, row in split_df.iterrows()}

label_lookup_train = build_label_lookup(rank_train_df)
label_lookup_val = build_label_lookup(rank_val_df)

# Encoders for user/item codes (fit on train + holdout users/items)
user_categories = pd.Index(sorted(set(train_df['user_id'].unique()) | set(holdout_df['user_id'].unique())))
item_categories = pd.Index(sorted(set(train_df['parent_asin'].unique()) | set(holdout_df['parent_asin'].unique())))

user_code_map = {u: i for i, u in enumerate(user_categories)}
item_code_map = {i: j for j, i in enumerate(item_categories)}

print(f"Users encoded: {len(user_code_map)}, Items encoded: {len(item_code_map)}")


Users encoded: 631986, Items encoded: 115709


In [6]:
def retrieve_candidates(model, trainset, user_id: str, items_pool: list, user_seen: dict, topN: int = 200):
    seen = user_seen.get(user_id, set())
    preds = []
    try:
        uid = trainset.to_inner_uid(user_id)
    except ValueError:
        return []
    for asin in items_pool:
        if asin in seen:
            continue
        try:
            iid = trainset.to_inner_iid(asin)
            est = model.predict(uid, iid).est
            preds.append((asin, est))
        except ValueError:
            continue
    preds.sort(key=lambda x: x[1], reverse=True)
    return preds[:topN]


def build_candidates(split_df: pd.DataFrame, label_lookup: dict, split_name: str):
    np.random.seed(42)
    records = []
    mf_global = mf_trainset.global_mean
    for user_id in split_df['user_id'].unique():
        top_candidates = retrieve_candidates(
            mf_model,
            mf_trainset,
            user_id=user_id,
            items_pool=items_pool,
            user_seen=user_seen_train,
            topN=CONFIG['retrieval']['topN'],
        )
        mf_est_map = dict(top_candidates)  # asin -> mf_est
        top_asins = list(mf_est_map.keys())

        remaining_pool = list(set(items_pool) - set(top_asins))
        n_random = min(CONFIG['retrieval']['n_random_neg'], len(remaining_pool))
        random_neg = np.random.choice(remaining_pool, size=n_random, replace=False) if n_random > 0 else []

        candidate_set = list(set(top_asins) | set(random_neg))

        for asin in candidate_set:
            label = label_lookup.get((user_id, asin), 0.0)
            mf_est = mf_est_map.get(asin, mf_global)
            records.append({
                'user_id': user_id,
                'parent_asin': asin,
                'label': label,
                'mf_est': mf_est,
            })
    df_candidates = pd.DataFrame(records)
    print(f"[{split_name}] candidates: {df_candidates.shape}")
    return df_candidates



In [7]:
print("Building candidate sets...")
train_candidates = build_candidates(rank_train_df, label_lookup_train, split_name="rank-train")
val_candidates = build_candidates(rank_val_df, label_lookup_val, split_name="rank-val")

# Attach stats
train_candidates = train_candidates.merge(user_stats, on='user_id', how='left')
train_candidates = train_candidates.merge(item_stats, on='parent_asin', how='left')

val_candidates = val_candidates.merge(user_stats, on='user_id', how='left')
val_candidates = val_candidates.merge(item_stats, on='parent_asin', how='left')

# Fill missing stats with zeros
for col in ['user_num_ratings', 'user_mean_rating', 'item_num_ratings', 'item_mean_rating']:
    train_candidates[col] = train_candidates[col].fillna(0)
    val_candidates[col] = val_candidates[col].fillna(0)

# Encode ids
train_candidates['user_code'] = train_candidates['user_id'].map(user_code_map)
train_candidates['item_code'] = train_candidates['parent_asin'].map(item_code_map)
val_candidates['user_code'] = val_candidates['user_id'].map(user_code_map)
val_candidates['item_code'] = val_candidates['parent_asin'].map(item_code_map)

# Features and labels
feature_cols = [
    'mf_est',
    'user_num_ratings', 'user_mean_rating',
    'item_num_ratings', 'item_mean_rating',
    'user_code', 'item_code',
]

X_train = train_candidates[feature_cols]
y_train = train_candidates['label']
X_val = val_candidates[feature_cols]
y_val = val_candidates['label']

print(f"Train candidates: {X_train.shape}, Val candidates: {X_val.shape}")



Building candidate sets...
[rank-train] candidates: (4281160, 4)
[rank-val] candidates: (4267120, 4)
Train candidates: (4281160, 7), Val candidates: (4267120, 7)


In [8]:
# Sort by user for grouping
train_candidates = train_candidates.sort_values('user_id').reset_index(drop=True)
val_candidates = val_candidates.sort_values('user_id').reset_index(drop=True)

X_train = train_candidates[feature_cols]
y_train = train_candidates['label']
X_val = val_candidates[feature_cols]
y_val = val_candidates['label']

group_train = train_candidates.groupby('user_id').size().tolist()
group_val = val_candidates.groupby('user_id').size().tolist()

print(f"Groups train: {len(group_train)}, Groups val: {len(group_val)}")

Groups train: 65864, Groups val: 65648


In [13]:
# Train XGBRanker
print("Training XGBRanker...")
ranker = xgb.XGBRanker(
    objective=CONFIG['ranker']['objective'],
    eval_metric=CONFIG['ranker']['eval_metric'],
    learning_rate=CONFIG['ranker']['learning_rate'],
    n_estimators=CONFIG['ranker']['n_estimators'],
    max_depth=CONFIG['ranker']['max_depth'],
    min_child_weight=CONFIG['ranker']['min_child_weight'],
    subsample=CONFIG['ranker']['subsample'],
    colsample_bytree=CONFIG['ranker']['colsample_bytree'],
    reg_lambda=CONFIG['ranker']['reg_lambda'],
    random_state=CONFIG['ranker']['random_state'],
    n_jobs=-1,
)

ranker.fit(
    X_train, y_train,
    group=group_train,
    eval_set=[(X_val, y_val)],
    eval_group=[group_val],
    verbose=50,
    # early_stopping_rounds=CONFIG['ranker']['early_stopping_rounds'],
    
)

print("Ranker training complete!")


Training XGBRanker...
[0]	validation_0-ndcg@10:0.98512
[50]	validation_0-ndcg@10:0.98536
[100]	validation_0-ndcg@10:0.98534
[150]	validation_0-ndcg@10:0.98535
[200]	validation_0-ndcg@10:0.98533
[250]	validation_0-ndcg@10:0.98536
[300]	validation_0-ndcg@10:0.98532
[350]	validation_0-ndcg@10:0.98527
[400]	validation_0-ndcg@10:0.98522
[450]	validation_0-ndcg@10:0.98517
[500]	validation_0-ndcg@10:0.98513
[550]	validation_0-ndcg@10:0.98508
[600]	validation_0-ndcg@10:0.98502
[650]	validation_0-ndcg@10:0.98497
[700]	validation_0-ndcg@10:0.98492
[750]	validation_0-ndcg@10:0.98489
[800]	validation_0-ndcg@10:0.98485
[850]	validation_0-ndcg@10:0.98487
[900]	validation_0-ndcg@10:0.98481
[950]	validation_0-ndcg@10:0.98479
[1000]	validation_0-ndcg@10:0.98480
[1050]	validation_0-ndcg@10:0.98479
[1100]	validation_0-ndcg@10:0.98474
[1150]	validation_0-ndcg@10:0.98471
[1200]	validation_0-ndcg@10:0.98466
[1250]	validation_0-ndcg@10:0.98467
[1300]	validation_0-ndcg@10:0.98462
[1350]	validation_0-ndcg@10:0

KeyboardInterrupt: 

In [11]:
# import numpy as np
# import xgboost as xgb

# --- ensure numpy ---
X_train_np = np.asarray(X_train)
y_train_np = np.asarray(y_train).astype(float)

X_val_np = np.asarray(X_val)
y_val_np = np.asarray(y_val).astype(float)

group_train_np = np.asarray(group_train).astype(int)
group_val_np   = np.asarray(group_val).astype(int)

# sanity checks
assert group_train_np.sum() == len(X_train_np)
assert group_val_np.sum() == len(X_val_np)

# --- build DMatrix with group info (required for ranking) ---
dtrain = xgb.DMatrix(X_train_np, label=y_train_np)
dtrain.set_group(group_train_np)

dval = xgb.DMatrix(X_val_np, label=y_val_np)
dval.set_group(group_val_np)

params = {
    "objective": CONFIG["ranker"]["objective"],      # e.g. "rank:pairwise"
    "eval_metric": CONFIG["ranker"]["eval_metric"],  # e.g. "ndcg@10"
    "learning_rate": CONFIG["ranker"]["learning_rate"],
    "max_depth": CONFIG["ranker"]["max_depth"],
    "min_child_weight": CONFIG["ranker"]["min_child_weight"],
    "subsample": CONFIG["ranker"]["subsample"],
    "colsample_bytree": CONFIG["ranker"]["colsample_bytree"],
    "lambda": CONFIG["ranker"]["reg_lambda"],
    "seed": CONFIG["ranker"]["random_state"],
    "nthread": -1,
    "tree_method": "hist",
}

print("Training xgb.train ranker...")
booster = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=CONFIG["ranker"]["n_estimators"],
    evals=[(dtrain, "train"), (dval, "val")],
    early_stopping_rounds=CONFIG["ranker"]["early_stopping_rounds"],
    verbose_eval=50,
)
print("Ranker training complete!")


Training xgb.train ranker...
[0]	train-ndcg@10:0.98519	val-ndcg@10:0.98512
[50]	train-ndcg@10:0.98540	val-ndcg@10:0.98536
[100]	train-ndcg@10:0.98570	val-ndcg@10:0.98534
[110]	train-ndcg@10:0.98574	val-ndcg@10:0.98534
Ranker training complete!


In [ ]:
# Evaluate NDCG@10
train_ndcg = ndcg_score([y_train], [ranker.predict(X_train)], k=10)
val_ndcg = ndcg_score([y_val], [ranker.predict(X_val)], k=10)
print(f"Train NDCG@10: {train_ndcg:.4f}")
print(f"Val NDCG@10: {val_ndcg:.4f}")

# Save artifacts
artifact = {
    'ranker': ranker,
    'feature_cols': feature_cols,
    'user_code_map': user_code_map,
    'item_code_map': item_code_map,
    'user_stats': user_stats,
    'item_stats': item_stats,
    'mf_model': mf_model,
    'mf_trainset': mf_trainset,
    'config': CONFIG,
    'items_pool': items_pool,
}

model_path = MODELS_DIR / "xgb_ranker.joblib"
joblib.dump(artifact, model_path)
print(f"Saved ranker artifacts to: {model_path}")

# Inference helper
def recommend(user_id: str, k: int = 10):
    candidates = retrieve_candidates(
        mf_model,
        mf_trainset,
        user_id=user_id,
        items_pool=items_pool,
        user_seen=user_seen_train,
        topN=CONFIG['retrieval']['topN'],
    )
    candidate_asins = [asin for asin, _ in candidates]

    df_tmp = pd.DataFrame({
        'user_id': [user_id] * len(candidate_asins),
        'parent_asin': candidate_asins,
    })
    df_tmp['mf_est'] = [pred for _, pred in candidates]

    df_tmp = df_tmp.merge(user_stats, on='user_id', how='left')
    df_tmp = df_tmp.merge(item_stats, on='parent_asin', how='left')
    for col in ['user_num_ratings', 'user_mean_rating', 'item_num_ratings', 'item_mean_rating']:
        df_tmp[col] = df_tmp[col].fillna(0)

    df_tmp['user_code'] = df_tmp['user_id'].map(user_code_map)
    df_tmp['item_code'] = df_tmp['parent_asin'].map(item_code_map)

    X_cand = df_tmp[feature_cols]
    scores = ranker.predict(X_cand)
    df_tmp['rank_score'] = scores
    df_tmp = df_tmp.sort_values('rank_score', ascending=False).head(k)
    return df_tmp[['parent_asin', 'rank_score', 'mf_est']]

# Quick smoke test
sample_user = rank_train_df['user_id'].iloc[0]
print(f"Running recommend() for user {sample_user}...")
res = recommend(sample_user, k=5)
print(res)


TypeError: ('Expecting data to be a DMatrix object, got: ', <class 'pandas.core.frame.DataFrame'>)

In [15]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import ndcg_score

# Build DMatrix (same as training)
dtrain = xgb.DMatrix(np.asarray(X_train), label=np.asarray(y_train).astype(float))
dtrain.set_group(np.asarray(group_train).astype(int))

dval = xgb.DMatrix(np.asarray(X_val), label=np.asarray(y_val).astype(float))
dval.set_group(np.asarray(group_val).astype(int))

train_pred = booster.predict(dtrain)
val_pred = booster.predict(dval)


In [16]:
def mean_ndcg_at_k(y_true, y_score, group, k=10):
    y_true = np.asarray(y_true).astype(float)
    y_score = np.asarray(y_score).astype(float)
    group = np.asarray(group).astype(int)

    scores = []
    start = 0
    for g in group:
        end = start + g
        yt = y_true[start:end]
        ys = y_score[start:end]

        # skip tiny groups or groups with no positives (optional, but common)
        if len(yt) == 0:
            start = end
            continue

        scores.append(ndcg_score([yt], [ys], k=min(k, len(yt))))
        start = end

    return float(np.mean(scores)) if scores else 0.0

train_ndcg = mean_ndcg_at_k(y_train, train_pred, group_train, k=10)
val_ndcg = mean_ndcg_at_k(y_val, val_pred, group_val, k=10)

print(f"Train mean NDCG@10: {train_ndcg:.4f}")
print(f"Val mean NDCG@10: {val_ndcg:.4f}")


Train mean NDCG@10: 0.0077
Val mean NDCG@10: 0.0068


In [17]:
import joblib

# Save booster
xgb_path = MODELS_DIR / "xgb_ranker.json"
booster.save_model(xgb_path)

artifact = {
    "xgb_model_path": str(xgb_path),
    "feature_cols": feature_cols,
    "user_code_map": user_code_map,
    "item_code_map": item_code_map,
    "user_stats": user_stats,
    "item_stats": item_stats,
    "mf_model": mf_model,
    "mf_trainset": mf_trainset,
    "config": CONFIG,
    "items_pool": items_pool,
}

meta_path = MODELS_DIR / "ranker_artifacts.joblib"
joblib.dump(artifact, meta_path)
print(f"Saved: {xgb_path} and {meta_path}")


Saved: /Users/charukagunawardhane/Documents/Development/ML_DL/LocalEnv-recommendation-system/models/xgb_ranker.json and /Users/charukagunawardhane/Documents/Development/ML_DL/LocalEnv-recommendation-system/models/ranker_artifacts.joblib


In [18]:
def recommend(user_id: str, k: int = 10):
    candidates = retrieve_candidates(
        mf_model,
        mf_trainset,
        user_id=user_id,
        items_pool=items_pool,
        user_seen=user_seen_train,
        topN=CONFIG["retrieval"]["topN"],
    )
    candidate_asins = [asin for asin, _ in candidates]
    if len(candidate_asins) == 0:
        return pd.DataFrame(columns=["parent_asin", "rank_score", "mf_est"])

    df_tmp = pd.DataFrame({
        "user_id": [user_id] * len(candidate_asins),
        "parent_asin": candidate_asins,
        "mf_est": [pred for _, pred in candidates],
    })

    df_tmp = df_tmp.merge(user_stats, on="user_id", how="left")
    df_tmp = df_tmp.merge(item_stats, on="parent_asin", how="left")

    for col in ["user_num_ratings", "user_mean_rating", "item_num_ratings", "item_mean_rating"]:
        df_tmp[col] = df_tmp[col].fillna(0)

    df_tmp["user_code"] = df_tmp["user_id"].map(user_code_map).fillna(-1).astype(int)
    df_tmp["item_code"] = df_tmp["parent_asin"].map(item_code_map).fillna(-1).astype(int)

    X_cand = df_tmp[feature_cols].to_numpy()
    dtest = xgb.DMatrix(X_cand)

    scores = booster.predict(dtest)
    df_tmp["rank_score"] = scores

    return df_tmp.sort_values("rank_score", ascending=False).head(k)[["parent_asin", "rank_score", "mf_est"]]


# Quick smoke test
sample_user = rank_train_df['user_id'].iloc[0]
print(f"Running recommend() for user {sample_user}...")
res = recommend(sample_user, k=5)
print(res)

Running recommend() for user AFVSNH37FEVCO6AKXMXHPFXEGNRQ...
   parent_asin  rank_score    mf_est
33  B07TK2PSJF    1.968661  3.960245
3   B07C533XCW    1.411686  3.960245
7   B0092MCQZ4    0.626208  3.960245
0   B007IAE5WY    0.619124  3.960245
6   B0719KWG8H    0.617749  3.960245


In [ ]:
# Evaluate
y_train_pred = model.predict(X_train)
y_val_pred = model.predict(X_val)

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))

print(f"Train RMSE: {train_rmse:.4f}")
print(f"Validation RMSE: {val_rmse:.4f}")


TypeError: ('Expecting data to be a DMatrix object, got: ', <class 'pandas.core.frame.DataFrame'>)

In [22]:
from sklearn.metrics import ndcg_score
import numpy as np

def mean_ndcg_at_k(y_true, y_pred, group, k=10):
    scores = []
    start = 0
    for g in group:
        end = start + g
        yt = y_true[start:end]
        yp = y_pred[start:end]

        if len(yt) > 1:
            scores.append(ndcg_score([yt], [yp], k=min(k, len(yt))))
        start = end
    return float(np.mean(scores))

train_ndcg = mean_ndcg_at_k(y_train, y_train_pred, group_train, k=10)
val_ndcg   = mean_ndcg_at_k(y_val, y_val_pred, group_val, k=10)

print(f"Train NDCG@10: {train_ndcg:.4f}")
print(f"Val   NDCG@10: {val_ndcg:.4f}")


Train NDCG@10: 0.0077
Val   NDCG@10: 0.0068


In [ ]:
# Save pipeline (preprocessor + model)
pipeline_path = MODELS_DIR / "xgb_ranker.joblib"

pipeline = {
    'preprocessor': preprocessor,
    'model': model,
    'numeric_features': available_numeric,
    'categorical_features': available_categorical
}

joblib.dump(pipeline, pipeline_path)

print(f"Saved pipeline to: {pipeline_path}")
print(f"Pipeline file size: {pipeline_path.stat().st_size / (1024*1024):.2f} MB")
print("\nRanking model training complete!")


## 1. Save the XGBoost Ranker

In [23]:
xgb_model_path = MODELS_DIR / "xgb_ranker.json"
booster.save_model(xgb_model_path)

print(f"Saved XGBoost ranker to: {xgb_model_path}")


Saved XGBoost ranker to: /Users/charukagunawardhane/Documents/Development/ML_DL/LocalEnv-recommendation-system/models/xgb_ranker.json


## 2. Save the rest of the pipeline (joblib-safe)

In [26]:
pipeline_path = MODELS_DIR / "ranking_pipeline.joblib"

pipeline = {
    # feature definition (THIS is what actually matters)
    "feature_cols": feature_cols,

    # maps & stats used at inference
    "user_code_map": user_code_map,
    "item_code_map": item_code_map,
    "user_stats": user_stats,
    "item_stats": item_stats,

    # retrieval stage artifacts
    "mf_model": mf_model,
    "mf_trainset": mf_trainset,
    "items_pool": items_pool,

    # ranker pointer
    "xgb_model_path": str(xgb_model_path),

    # config
    "config": CONFIG,
}

joblib.dump(pipeline, pipeline_path)

print(f"Saved pipeline metadata to: {pipeline_path}")
print(f"Pipeline file size: {pipeline_path.stat().st_size / (1024*1024):.2f} MB")
print("\nRanking model training complete!")



Saved pipeline metadata to: /Users/charukagunawardhane/Documents/Development/ML_DL/LocalEnv-recommendation-system/models/ranking_pipeline.joblib
Pipeline file size: 388.90 MB

Ranking model training complete!
